# 10과 - 프로덕션 AI 에이전트

이 과에서는 Microsoft Agent Framework (Python)를 사용한 AI 에이전트의 <strong>프로덕션 패턴</strong>을 배웁니다. 다루는 내용은:

- **가시성(Observability)** — 에이전트 상호작용에 타이밍과 로깅 추가
- **평가(Evaluation)** — 평가 에이전트를 사용해 응답 품질 점수화
- **비용 관리(Cost management)** — 토큰 최적화 및 모델 선택 전략

시나리오는 사용자의 여행 계획을 돕는 <strong>여행 에이전트</strong>이며, 그 위에 모니터링과 평가가 겹쳐져 있습니다.


## 설정


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## 프로덕션 고려사항

AI 에이전트를 프로토타입에서 프로덕션으로 이전하려면 세 가지 핵심 요소에 주의를 기울여야 합니다:

1. **관찰 가능성(Observability)** — 에이전트가 무엇을 하는지, 얼마나 시간이 걸리는지, 어떤 도구를 호출하는지에 대한 가시성이 필요합니다. 추적 및 로깅 없이는 프로덕션 문제를 디버깅하는 것이 거의 불가능합니다.

2. **평가(Evaluation)** — 자동화된 품질 검사는 시간이 지나도 에이전트의 응답이 정확하고 완전하며 도움이 되는 상태를 유지하도록 합니다. 평가자 에이전트는 정의된 기준에 따라 응답에 점수를 매길 수 있습니다.

3. **비용 관리(Cost Management)** — 토큰 사용량은 비용에 직접적인 영향을 미칩니다. 프롬프트 최적화, 모델 선택, 캐싱 등의 전략은 품질을 희생하지 않고 비용을 관리하는 데 도움이 됩니다.


## 관찰 가능한 에이전트 구축하기

우리는 여행 도구를 정의하고 에이전트 호출을 타이밍으로 감싸서 대기 시간을 모니터링할 수 있게 합니다. 운영 환경에서는 OpenTelemetry나 유사한 추적 백엔드와 통합할 것입니다.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## 평가 패턴

일반적인 생산 패턴은 두 번째 에이전트를 <strong>평가자</strong>로 사용하는 것입니다. 평가자는 완성도, 정확성 및 유용성과 같은 미리 정의된 기준에 따라 기본 에이전트의 응답을 평가합니다.

이것은 다음을 가능하게 합니다:
- 사용자에게 응답이 도달하기 전에 자동화된 품질 게이트
- 프롬프트나 모델이 변경될 때 회귀 감지
- 시간 경과에 따른 에이전트 성능의 지속적인 모니터링


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## 비용 관리 전략

비용 통제는 생산 AI 에이전트에 매우 중요합니다. 주요 전략은 다음과 같습니다:

| 전략 | 설명 |
|---|---|
| **프롬프트 최적화** | 시스템 지침을 간결하게 유지하세요. 중복된 컨텍스트를 제거하여 입력 토큰 수를 줄입니다. |
    "| **모델 선택** | 분류나 추출과 같은 간단한 작업에는 더 작고 저렴한 모델(e.g. GPT-4.1-mini)을 사용하고, 복잡한 추론에는 더 큰 모델을 예약해 사용하세요. |\n",
| <strong>캐싱</strong> | 도구 결과와 자주 하는 쿼리를 캐시하여 중복된 API 호출을 피하세요. |
| **토큰 예산** | 예상치 못하게 긴 응답을 방지하기 위해 `max_tokens` 제한을 설정하세요. |
| **배치 처리** | 가능한 경우 여러 사용자 쿼리를 하나의 API 호출로 그룹화하세요. |

실제로는 계층적 접근이 잘 작동합니다: 단순한 요청은 빠르고 저렴한 모델로 보내고, 복잡한 쿼리만 더 강력한 모델로 올립니다.


## 요약

이 수업에서 배운 내용은 다음과 같습니다:

1. **타이밍과 로깅을 통해** 에이전트 상호작용에 관측 가능성을 추가하여 추적 및 모니터링의 기초를 마련하는 방법.
2. **평가자 에이전트를 사용하여** 응답의 완전성, 정확성, 유용성을 자동으로 평가하는 방법.
3. **프롬프트 최적화, 모델 선택, 캐싱, 토큰 예산 관리를 통해** 비용을 관리하는 방법.

이러한 프로덕션 패턴은 AI 에이전트를 신뢰할 수 있고, 측정 가능하며, 대규모로 비용 효율적으로 만드는 데 도움이 됩니다.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**면책 조항**:
이 문서는 AI 번역 서비스 [Co-op Translator](https://github.com/Azure/co-op-translator)를 사용하여 번역되었습니다. 정확성을 기하기 위해 노력하고 있으나, 자동 번역은 오류나 부정확한 부분이 있을 수 있음을 유의하시기 바랍니다. 원본 문서의 원어본이 권위 있는 자료로 간주되어야 합니다. 중요한 정보의 경우, 전문가의 인간 번역을 권장합니다. 이 번역 사용으로 인해 발생하는 오해나 잘못된 해석에 대해 당사는 책임을 지지 않습니다.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
